# Tiền xử lý dữ liệu IDS — Camera AI ONT

Notebook này thực hiện **hai bước làm sạch bắt buộc** trước khi train Decision Tree:

| Bước | Nội dung | Lý do |
|---|---|---|
| **1** | Gán lại nhãn ở **mức flow** thay vì mức file | `pcap_to_csv.py` ghi cùng một `--label` cho mọi flow trong file, nên `local_scanport_flows.csv` chứa cả traffic nền bị gán nhầm là `scanport` |
| **2** | Loại 3 feature bị **độ dài phiên capture** chi phối | Hai file được bắt trong hai cửa sổ dài ngắn khác nhau (25,2 phút vs 5,4 phút), khiến `duration` / `active_min` / `active_std` mang tín hiệu giả của phiên bắt gói chứ không phải hành vi |

Đầu ra: `dataset_csv/ids_dataset_labeled.csv` — dữ liệu đã gán nhãn đúng, kèm cột metadata để truy vết.

In [ ]:
# Cấu hình số luồng CPU trước khi import NumPy/Pandas/TensorFlow.
import os

logical_cpus = os.cpu_count() or 1
compute_threads = max(1, logical_cpus // 2)
interop_threads = min(8, max(1, logical_cpus // 4))

os.environ['OPENBLAS_NUM_THREADS'] = str(compute_threads)
os.environ['MKL_NUM_THREADS'] = str(compute_threads)
os.environ['OMP_NUM_THREADS'] = str(compute_threads)
os.environ['NUMEXPR_NUM_THREADS'] = str(compute_threads)
os.environ['TF_NUM_INTRAOP_THREADS'] = str(compute_threads)
os.environ['TF_NUM_INTEROP_THREADS'] = str(interop_threads)

# Chỉ bỏ comment nếu cần tắt oneDNN để kiểm tra sai khác số học.
# os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'

print(f'CPU logic: {logical_cpus}')
print(f'BLAS/TF intra-op: {compute_threads} luồng')
print(f'TF inter-op: {interop_threads} luồng')


CPU logic: 12 | BLAS: 6 luồng


In [2]:
# =========================================================
# BƯỚC 0: Import
# Decision Tree không cần scaling (bất biến với biến đổi đơn điệu)
# và không cần TensorFlow -> thư viện gọn hơn nhiều so với bản NN.
# =========================================================
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 60)

BASE_DIR    = Path.cwd().parent if Path.cwd().name == 'Decesion_trees_train_model' else Path.cwd()
DATASET_DIR = BASE_DIR / 'dataset_csv'

NORMAL_CSV  = DATASET_DIR / 'local_normal_flows.csv'
ATTACK_CSV  = DATASET_DIR / 'local_scanport_flows.csv'
FEATURE_XLS = DATASET_DIR / 'feature_selection_IDS.xlsx'
OUT_CSV     = DATASET_DIR / 'ids_dataset_labeled.csv'

for p in (NORMAL_CSV, ATTACK_CSV, FEATURE_XLS):
    assert p.exists(), f'Thiếu file: {p}'
print(f'Thư mục dữ liệu: {DATASET_DIR}')

Thư mục dữ liệu: D:\01.AI_Security\Camera_AI_ONT\dataset_csv


In [ ]:
# =========================================================
# Đọc danh sách feature TRỰC TIẾP từ file Excel
# =========================================================
fs = pd.read_excel(FEATURE_XLS, sheet_name='Feature Selection')
fs.columns = [str(c).strip() for c in fs.columns]

KEEP_FEATURES = (
    fs.loc[fs['Quyết định'].astype(str).str.strip() == '"GIỮ"', 'Feature']
      .astype(str).str.strip().tolist()
)

print(f'Feature GIỮ trong Excel: {len(KEEP_FEATURES)}')
print(KEEP_FEATURES)

Feature GIỮ trong Excel: 27
['flow_pkts_per_s', 'bwd_pkts_per_s', 'flow_bytes_per_s', 'tot_pkts', 'flow_iat_min', 'flow_iat_mean', 'fwd_iat_min', 'fwd_iat_mean', 'fwd_iat_std', 'bwd_iat_mean', 'duration', 'active_min', 'active_std', 'idle_std', 'pkt_len_min', 'pkt_len_max', 'pkt_len_mean', 'pkt_len_std', 'fwd_pkt_len_std', 'bwd_pkt_len_mean', 'syn_cnt', 'rst_cnt', 'fin_cnt', 'down_up_pkt_ratio', 'down_up_byte_ratio', 'bwd_payload_bytes', 'bwd_pkts_with_payload']


In [4]:
# =========================================================
# Đọc 2 file flow-level
# =========================================================
df_normal = pd.read_csv(NORMAL_CSV)
df_attack = pd.read_csv(ATTACK_CSV)

print(f'local_normal_flows.csv   : {df_normal.shape[0]:>5} flow, {df_normal.shape[1]} cột')
print(f'local_scanport_flows.csv : {df_attack.shape[0]:>5} flow, {df_attack.shape[1]} cột')

missing = [f for f in KEEP_FEATURES if f not in df_normal.columns]
assert not missing, f'Feature trong Excel không có trong CSV: {missing}'
print('\nTất cả feature trong Excel đều tồn tại trong CSV.')

# Cửa sổ capture của từng file -> căn cứ cho Bước 2
for name, d in [('normal', df_normal), ('scanport', df_attack)]:
    span = d['end_time'].max() - d['start_time'].min()
    ts   = pd.to_datetime(d['start_time'].min(), unit='s')
    print(f'{name:9s}: bắt đầu {ts:%Y-%m-%d %H:%M:%S} | dài {span/60:6.1f} phút '
          f'| {len(d)/(span/60):7.1f} flow/phút')

local_normal_flows.csv   :  1927 flow, 79 cột
local_scanport_flows.csv :  4501 flow, 79 cột

Tất cả feature trong Excel đều tồn tại trong CSV.
normal   : bắt đầu 2026-08-14 02:01:34 | dài   25.2 phút |    76.6 flow/phút
scanport : bắt đầu 2026-08-14 06:33:07 | dài    5.4 phút |   837.8 flow/phút


---
## BƯỚC 1 — Gán lại nhãn ở mức flow

### Vấn đề

`pcap_to_csv.py` nhận `--label` từ dòng lệnh rồi ghi **cùng một giá trị đó vào mọi flow** của file
(xem `Flow.row(label)`, dòng 253 và các lời gọi ở dòng 442/458/488). Nghĩa là mọi gói tin lọt vào
cửa sổ 5,4 phút chạy nmap đều bị đóng dấu `scanport`, kể cả traffic HTTPS/DNS/NTP/SSDP hoàn toàn
bình thường chạy nền lúc đó.

### Cách xác định flow scan thật

Port scan để lại một dấu vết **cross-flow** đặc trưng: một nguồn bắn vào một đích với **rất nhiều
cổng khác nhau**. Ta dùng chính đặc điểm đó (fan-out) để tìm hướng tấn công một cách **tự động từ
dữ liệu**, thay vì hardcode địa chỉ IP — như vậy notebook vẫn chạy đúng với các phiên capture sau.

In [5]:
# =========================================================
# 1.1 — Phát hiện hướng tấn công bằng fan-out số cổng đích
# =========================================================
FANOUT_THRESHOLD = 100   # số cổng đích tối thiểu để coi là scan

pair_stats = (
    df_attack.groupby(['src_ip', 'dst_ip'])
             .agg(n_flows=('flow_id', 'size'),
                  n_dst_ports=('dst_port', 'nunique'),
                  n_syn=('syn_cnt', 'sum'),
                  median_pkts=('tot_pkts', 'median'))
             .sort_values('n_dst_ports', ascending=False)
)

print('Top cặp (src_ip -> dst_ip) theo số cổng đích khác nhau:')
print(pair_stats.head(8).to_string())

candidates = pair_stats[pair_stats['n_dst_ports'] >= FANOUT_THRESHOLD]
assert len(candidates) > 0, 'Không tìm thấy cặp nào có fan-out đủ lớn.'

ATTACKER_IP, TARGET_IP = candidates.index[0]     # cặp fan-out lớn nhất = hướng tấn công
print(f'\n>>> Attacker : {ATTACKER_IP}')
print(f'>>> Target   : {TARGET_IP}')
print(f'>>> Fan-out  : {candidates.iloc[0]["n_dst_ports"]:.0f} cổng đích / '
      f'{candidates.iloc[0]["n_flows"]:.0f} flow')

Top cặp (src_ip -> dst_ip) theo số cổng đích khác nhau:
                                                     n_flows  n_dst_ports  n_syn  median_pkts
src_ip                    dst_ip                                                             
192.168.2.8               192.168.2.168                 2820         1000   2945          2.0
192.168.2.168             192.168.2.8                    404          116      2          1.0
192.168.2.91              103.238.69.131                  20           17      0          3.0
142.250.198.163           192.168.2.91                    11           11      0          1.0
35.244.180.134            192.168.2.52                    12            9      2          1.5
fe80::a6f4:c2ff:fe0b:92b5 fe80::b893:6eff:fe6b:5b26        6            6      0          1.0
35.186.224.24             192.168.2.91                     5            5      0          2.0
52.123.131.14             192.168.2.91                     5            5      0          1.0

>>>

In [6]:
# =========================================================
# 1.2 — Chia file attack thành 3 nhóm
# =========================================================
is_scan     = (df_attack['src_ip'] == ATTACKER_IP) & (df_attack['dst_ip'] == TARGET_IP)
is_response = (df_attack['src_ip'] == TARGET_IP)   & (df_attack['dst_ip'] == ATTACKER_IP)
is_backgrnd = ~is_scan & ~is_response

print(f'A. Flow scan  (attacker -> target) : {is_scan.sum():>5}')
print(f'B. Flow phản hồi (target -> attacker): {is_response.sum():>5}')
print(f'C. Traffic nền (không liên quan)   : {is_backgrnd.sum():>5}')
print(f'{"":->44}\nTổng                                : {len(df_attack):>5}')

# Kiểm chứng nhóm A đúng là scan chứ không lẫn session thật
scan_rows = df_attack[is_scan]
print(f'\n--- Đặc điểm nhóm A ---')
print(f'Protocol      : {scan_rows["protocol"].value_counts().to_dict()}')
print(f'tot_pkts max  : {scan_rows["tot_pkts"].max()}  (session thật sẽ lớn hơn nhiều)')
print(f'tot_pkts phổ biến: {scan_rows["tot_pkts"].value_counts().head(3).to_dict()}')

resp_rows = df_attack[is_response]
print(f'\n--- Đặc điểm nhóm B ---')
print(f'tot_pkts == 1 : {(resp_rows["tot_pkts"] == 1).sum()} / {len(resp_rows)}')
print(f'syn_cnt  == 0 : {(resp_rows["syn_cnt"] == 0).sum()} / {len(resp_rows)}')

A. Flow scan  (attacker -> target) :  2820
B. Flow phản hồi (target -> attacker):   404
C. Traffic nền (không liên quan)   :  1277
--------------------------------------------
Tổng                                :  4501

--- Đặc điểm nhóm A ---
Protocol      : {'TCP': 2820}
tot_pkts max  : 10  (session thật sẽ lớn hơn nhiều)
tot_pkts phổ biến: {2: 2313, 1: 311, 7: 100}

--- Đặc điểm nhóm B ---
tot_pkts == 1 : 399 / 404
syn_cnt  == 0 : 402 / 404


### Xử lý từng nhóm

| Nhóm | Quyết định | Lý do |
|---|---|---|
| **A** — attacker → target | nhãn `scanport` | Hành vi tấn công thật sự cần phát hiện |
| **B** — target → attacker | **loại khỏi dataset** | Là gói RST phản hồi của camera bị flow-builder tách thành flow riêng (99% chỉ có 1 gói, không có SYN). Đây là *hệ quả* của scan, không phải hành vi của kẻ tấn công. Gán `scanport` sẽ dạy model coi nạn nhân là thủ phạm; gán `normal` lại dạy model bỏ qua chuỗi RST bất thường. Loại bỏ là lựa chọn ít sai lệch nhất. |
| **C** — traffic nền | nhãn `normal` | Là HTTPS/DNS/NTP/SSDP bình thường, chỉ vô tình bị bắt trong cửa sổ tấn công |

Việc chuyển nhóm C sang `normal` còn có lợi ích phụ quan trọng: lớp `normal` giờ chứa dữ liệu từ
**cả hai phiên capture** (09:01 và 13:33), nên model không thể dùng đặc điểm riêng của một phiên
để phân biệt hai lớp.

Đổi `RESPONSE_POLICY` nếu muốn thử phương án khác.

In [7]:
# =========================================================
# 1.3 — Áp dụng nhãn mới
# =========================================================
RESPONSE_POLICY = 'exclude'      # 'exclude' | 'normal' | 'scanport'

parts = [
    df_normal.assign(label_new='normal',   source='normal_capture',  flow_role='background'),
    df_attack[is_backgrnd].assign(label_new='normal', source='attack_capture', flow_role='background'),
    df_attack[is_scan].assign(label_new='scanport',   source='attack_capture', flow_role='scan'),
]

if RESPONSE_POLICY != 'exclude':
    parts.append(
        df_attack[is_response].assign(label_new=RESPONSE_POLICY,
                                      source='attack_capture', flow_role='response')
    )

df = pd.concat(parts, ignore_index=True)

print(f'Chính sách với flow phản hồi: {RESPONSE_POLICY}')
print(f'Dataset sau gán nhãn: {len(df)} flow\n')

print('--- So sánh nhãn cũ và nhãn mới ---')
print(pd.crosstab(df['label'], df['label_new'],
                  rownames=['nhãn cũ (theo file)'], colnames=['nhãn mới (theo flow)']).to_string())

n_changed = (df['label'] != df['label_new']).sum()
print(f'\nSố flow bị đổi nhãn: {n_changed} ({n_changed/len(df):.1%})')

Chính sách với flow phản hồi: exclude
Dataset sau gán nhãn: 6024 flow

--- So sánh nhãn cũ và nhãn mới ---
nhãn mới (theo flow)  normal  scanport
nhãn cũ (theo file)                   
normal                  1927         0
scanport                1277      2820

Số flow bị đổi nhãn: 1277 (21.2%)


---
## BƯỚC 2 — Loại feature bị độ dài phiên capture chi phối

### Vấn đề

Một flow không thể kéo dài hơn phiên bắt gói chứa nó. Vì hai file được bắt trong hai cửa sổ dài
ngắn rất khác nhau (25,2 phút vs 5,4 phút), giá trị **lớn nhất** của các feature đo thời gian bị
chặn ở hai mức khác nhau theo đúng độ dài phiên — không liên quan gì tới hành vi.

Cây quyết định hoàn toàn có thể học luật kiểu `duration > 322 → normal`: **đúng 100% trên dataset
này và vô nghĩa hoàn toàn ngoài thực tế**. Đây là dạng leakage mà file Excel chưa lường tới (Excel
mới chỉ cảnh báo nhóm định danh IP/port/timestamp).

Ô bên dưới **kiểm chứng** giả thuyết đó trước khi loại, thay vì loại theo cảm tính.

In [8]:
# =========================================================
# 2.1 — Kiểm chứng: feature nào chạm trần độ dài phiên capture?
# =========================================================
span_normal = df_normal['end_time'].max() - df_normal['start_time'].min()
span_attack = df_attack['end_time'].max() - df_attack['start_time'].min()
print(f'Độ dài phiên: normal = {span_normal:.2f}s | attack = {span_attack:.2f}s\n')

TIME_FEATURES = [f for f in KEEP_FEATURES
                 if any(k in f for k in ('duration', 'active', 'idle', 'iat'))]

rows = []
for f in TIME_FEATURES:
    mx_n = df_normal[f].max()
    mx_a = df_attack[f].max()
    rows.append({
        'feature': f,
        'max_normal': round(mx_n, 3),
        'max_attack': round(mx_a, 3),
        # chạm trần = giá trị max trùng khít độ dài phiên capture tương ứng
        'chạm_trần': np.isclose(mx_n, span_normal, rtol=1e-3) and np.isclose(mx_a, span_attack, rtol=1e-3),
    })

check = pd.DataFrame(rows).sort_values('chạm_trần', ascending=False)
print(check.to_string(index=False))

CAPTURE_BOUND = check.loc[check['chạm_trần'], 'feature'].tolist()
print(f'\n>>> Feature chạm trần phiên capture: {CAPTURE_BOUND}')

Độ dài phiên: normal = 1509.74s | attack = 322.33s

      feature  max_normal  max_attack  chạm_trần
     duration    1509.744     322.329       True
   active_min    1509.744     322.329       True
 flow_iat_min       7.103       8.533      False
flow_iat_mean      42.944      28.910      False
  fwd_iat_min      71.439      96.284      False
 fwd_iat_mean      71.439      96.284      False
  fwd_iat_std      51.643      46.343      False
 bwd_iat_mean      90.735      48.094      False
   active_std     112.191      15.626      False
     idle_std      36.955      31.448      False

>>> Feature chạm trần phiên capture: ['duration', 'active_min']


In [9]:
# =========================================================
# 2.2 — Loại feature artifact
# =========================================================
# active_std bổ sung thủ công: không chạm trần tuyệt đối nhưng biên độ lệch
# theo đúng tỉ lệ độ dài phiên (normal 112.2s vs attack 15.6s ~ 7x, sát tỉ lệ 4.7x
# của hai cửa sổ capture) -> vẫn là artifact chứ không phải hành vi.
DROP_FEATURES = sorted(set(CAPTURE_BOUND) | {'active_std'})

FINAL_FEATURES = [f for f in KEEP_FEATURES if f not in DROP_FEATURES]

print(f'Feature từ Excel      : {len(KEEP_FEATURES)}')
print(f'Loại (artifact capture): {len(DROP_FEATURES)} -> {DROP_FEATURES}')
print(f'Feature dùng để train : {len(FINAL_FEATURES)}\n')
for i, f in enumerate(FINAL_FEATURES, 1):
    print(f'{i:>2}. {f}')

Feature từ Excel      : 27
Loại (artifact capture): 3 -> ['active_min', 'active_std', 'duration']
Feature dùng để train : 24

 1. flow_pkts_per_s
 2. bwd_pkts_per_s
 3. flow_bytes_per_s
 4. tot_pkts
 5. flow_iat_min
 6. flow_iat_mean
 7. fwd_iat_min
 8. fwd_iat_mean
 9. fwd_iat_std
10. bwd_iat_mean
11. idle_std
12. pkt_len_min
13. pkt_len_max
14. pkt_len_mean
15. pkt_len_std
16. fwd_pkt_len_std
17. bwd_pkt_len_mean
18. syn_cnt
19. rst_cnt
20. fin_cnt
21. down_up_pkt_ratio
22. down_up_byte_ratio
23. bwd_payload_bytes
24. bwd_pkts_with_payload


In [10]:
# =========================================================
# 2.3 — Xác nhận không còn feature nào bị trần phiên capture
# =========================================================
residual = [f for f in FINAL_FEATURES
            if np.isclose(df[f].max(), span_normal, rtol=1e-3)
            or np.isclose(df[f].max(), span_attack, rtol=1e-3)]

if residual:
    print(f'CẢNH BÁO — vẫn còn feature chạm trần: {residual}')
else:
    print('OK — không feature nào còn chạm trần độ dài phiên capture.')

print(f'\nGiá trị lớn nhất của các feature thời gian còn lại:')
for f in [x for x in FINAL_FEATURES if any(k in x for k in ('iat', 'idle', 'active'))]:
    print(f'  {f:20s} max = {df[f].max():10.3f}s  (trần phiên: {span_attack:.1f}s / {span_normal:.1f}s)')

OK — không feature nào còn chạm trần độ dài phiên capture.

Giá trị lớn nhất của các feature thời gian còn lại:
  flow_iat_min         max =      8.533s  (trần phiên: 322.3s / 1509.7s)
  flow_iat_mean        max =     42.944s  (trần phiên: 322.3s / 1509.7s)
  fwd_iat_min          max =     96.284s  (trần phiên: 322.3s / 1509.7s)
  fwd_iat_mean         max =     96.284s  (trần phiên: 322.3s / 1509.7s)
  fwd_iat_std          max =     51.643s  (trần phiên: 322.3s / 1509.7s)
  bwd_iat_mean         max =     90.735s  (trần phiên: 322.3s / 1509.7s)
  idle_std             max =     36.955s  (trần phiên: 322.3s / 1509.7s)


---
## Kiểm tra chất lượng dữ liệu sau khi làm sạch

In [11]:
# =========================================================
# 3.1 — Báo cáo tổng hợp
# =========================================================
X_cols = FINAL_FEATURES

print('--- Phân bố lớp ---')
dist = df['label_new'].value_counts()
for k, v in dist.items():
    print(f'  {k:10s}: {v:>5} ({v/len(df):6.1%})')
print(f'  Tỉ lệ mất cân bằng: 1 : {dist.max()/dist.min():.2f}')

print('\n--- Nguồn dữ liệu của lớp normal (kiểm tra confound phiên capture) ---')
print(df[df['label_new'] == 'normal']['source'].value_counts().to_string())

print('\n--- Chất lượng cột ---')
print(f'  NaN            : {df[X_cols].isna().sum().sum()}')
print(f'  Inf            : {np.isinf(df[X_cols].to_numpy()).sum()}')
print(f'  Cột hằng số    : {[c for c in X_cols if df[c].nunique() <= 1]}')
print(f'  Cột non-numeric: {[c for c in X_cols if not pd.api.types.is_numeric_dtype(df[c])]}')

dup_all   = df.duplicated(subset=X_cols + ['label_new']).sum()
conflict  = (df.groupby(X_cols, dropna=False)['label_new'].nunique() > 1).sum()
print(f'\n--- Trùng lặp (xử lý ở bước tiếp theo, KHÔNG dedup ở đây) ---')
print(f'  Dòng trùng hoàn toàn      : {dup_all} / {len(df)} ({dup_all/len(df):.1%})')
print(f'  Nhóm mâu thuẫn nhãn       : {conflict}   <- trần lỗi không thể vượt qua')

--- Phân bố lớp ---
  normal    :  3204 ( 53.2%)
  scanport  :  2820 ( 46.8%)
  Tỉ lệ mất cân bằng: 1 : 1.14

--- Nguồn dữ liệu của lớp normal (kiểm tra confound phiên capture) ---
source
normal_capture    1927
attack_capture    1277

--- Chất lượng cột ---
  NaN            : 0
  Inf            : 0
  Cột hằng số    : []
  Cột non-numeric: []

--- Trùng lặp (xử lý ở bước tiếp theo, KHÔNG dedup ở đây) ---
  Dòng trùng hoàn toàn      : 1508 / 6024 (25.0%)
  Nhóm mâu thuẫn nhãn       : 3   <- trần lỗi không thể vượt qua


In [12]:
# =========================================================
# 3.2 — Sức phân tách của từng feature (AUC một chiều)
# So với TRƯỚC khi làm sạch: không feature nào vượt 0.70
# =========================================================
from sklearn.metrics import roc_auc_score

y = (df['label_new'] == 'scanport').astype(int)
auc = pd.Series({f: roc_auc_score(y, df[f]) for f in X_cols})
auc_strength = (auc - 0.5).abs() + 0.5     # AUC < 0.5 nghĩa là phân tách theo chiều ngược

report = (pd.DataFrame({'AUC': auc.round(4), 'sức_phân_tách': auc_strength.round(4)})
            .sort_values('sức_phân_tách', ascending=False))
print(report.to_string())
print(f'\nSố feature có sức phân tách > 0.80: {(auc_strength > 0.80).sum()} / {len(X_cols)}')

                          AUC  sức_phân_tách
pkt_len_mean           0.1882         0.8118
pkt_len_max            0.1927         0.8073
pkt_len_min            0.2254         0.7746
rst_cnt                0.7652         0.7652
syn_cnt                0.7488         0.7488
bwd_pkts_per_s         0.7295         0.7295
bwd_payload_bytes      0.3154         0.6846
bwd_pkts_with_payload  0.3154         0.6846
down_up_pkt_ratio      0.6769         0.6769
fwd_iat_mean           0.3299         0.6701
pkt_len_std            0.3426         0.6574
fwd_iat_min            0.3435         0.6565
fwd_pkt_len_std        0.3719         0.6281
bwd_pkt_len_mean       0.3807         0.6193
fwd_iat_std            0.3814         0.6186
flow_iat_min           0.6089         0.6089
down_up_byte_ratio     0.5897         0.5897
bwd_iat_mean           0.4126         0.5874
tot_pkts               0.4158         0.5842
fin_cnt                0.4382         0.5618
flow_pkts_per_s        0.5476         0.5476
flow_bytes

In [13]:
# =========================================================
# 3.3 — Lưu kết quả
# =========================================================
META_COLS = ['flow_id', 'src_ip', 'dst_ip', 'src_port', 'dst_port',
             'protocol', 'start_time', 'source', 'flow_role']

out = df[META_COLS + X_cols + ['label_new']].rename(columns={'label_new': 'label'})
out.to_csv(OUT_CSV, index=False)

print(f'Đã lưu: {OUT_CSV}')
print(f'  {len(out)} dòng x {out.shape[1]} cột')
print(f'  {len(META_COLS)} cột metadata (KHÔNG dùng để train — chỉ để truy vết)')
print(f'  {len(X_cols)} feature + 1 cột label')
print(f'\nPhân bố nhãn: {out["label"].value_counts().to_dict()}')

# Lưu kèm danh sách feature để notebook train không bị lệch thứ tự cột
import json
feature_meta = {
    'features': X_cols,
    'n_features': len(X_cols),
    'dropped_from_excel': DROP_FEATURES,
    'drop_reason': 'bị chặn bởi độ dài phiên capture (artifact, không phải hành vi)',
    'attacker_ip': ATTACKER_IP,
    'target_ip': TARGET_IP,
    'response_policy': RESPONSE_POLICY,
    'label_map': {'normal': 0, 'scanport': 1},
}
meta_path = DATASET_DIR / 'feature_meta.json'
meta_path.write_text(json.dumps(feature_meta, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'Đã lưu metadata: {meta_path}')

Đã lưu: D:\01.AI_Security\Camera_AI_ONT\dataset_csv\ids_dataset_labeled.csv
  6024 dòng x 34 cột
  9 cột metadata (KHÔNG dùng để train — chỉ để truy vết)
  24 feature + 1 cột label

Phân bố nhãn: {'normal': 3204, 'scanport': 2820}
Đã lưu metadata: D:\01.AI_Security\Camera_AI_ONT\dataset_csv\feature_meta.json


---
## Bước tiếp theo

Notebook này **cố ý dừng ở đây** — chưa dedup, chưa split, chưa scale. Những việc đó thuộc về
`model_train.ipynb` để ranh giới giữa hai notebook rõ ràng.

Việc còn lại theo kế hoạch:

1. ~~Gán lại nhãn ở mức flow~~ — xong
2. ~~Loại feature bị capture-length chi phối~~ — xong
3. **Capture thêm 3–4 phiên scan đa dạng** (`-T2` chậm, UDP scan, attacker khác, target khác) —
   đây là việc quyết định model có tổng quát hoá được hay không. Dataset hiện tại vẫn chỉ có
   **một** phiên nmap duy nhất.
4. Capture dữ liệu brute-force nếu giữ mục tiêu multi-class như thiết kế trong Excel
5. Dedup + split stratified + train Decision Tree

> **Lưu ý khi đọc kết quả train sắp tới:** điểm số sẽ rất cao (f1 ≈ 0,99). Con số đó phản ánh
> "nhận diện đúng lần nmap này", **không phải** năng lực phát hiện port scan nói chung. Chỉ sau
> khi có mục 3 mới kết luận được về khả năng tổng quát hoá.